# С этим товаром также покупают…

## Цель:
В этом задании вы продолжите работу с двухуровневой моделью, попробуете добавлять новые признаки, а также модели первого уровня, подберете гиперпараметры. В результате доработок качество модели, рассмотренной на занятии, должно улучшиться.


## Описание/Пошаговая инструкция выполнения домашнего задания:
Продолжим работу с данными онлайн-кинотеатра KION.
- Задача - улучшить качество двухуровневой модели, рассмотренной на занятии минимум на 10% по метрике precision@20. 
- Для этого предлагается проделать хотя бы два из трех шагов:
  - добавить одну или несколько моделей первого уровня (можно брать модели из библиотеки implicit или других библиотек); 
    - на основе предсказаний этих моделей отобрать объекты-кандидаты для модели второго уровня;
  - фича-инжиниринг: поработать с имеющимися характеристиками пользователей и объектов и/или сгенерировать новые фичи на основе популярности объектов/активности пользователей; попробовать учесть временную составляющую, например, считать популярность за разные временные промежутки.
  - подобрать лучшее train-val-test разбиение; подобрать гиперпараметры моделей первого и второго уровней.
- Оценить качество модели по метрикам precision@20, recall@20, mrr@20.
- Написать отчет о проделанной работе: что удалось реализовать; что сработало/не сработало; что дало наибольший прирост качество; любые другие дополнения, относящиеся к проделанной работе.

# Подготовка данных


In [28]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import implicit

In [29]:
interactions = pd.read_csv("interactions_df.csv")
items = pd.read_csv("items.csv")
users = pd.read_csv("users.csv")

interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])
print(interactions.shape, users.shape, items.shape)

(5476251, 5) (840197, 5) (15963, 14)


In [30]:
interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


In [31]:
items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [32]:
users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [33]:

MIN_DUR = 300                 # минимальная длительность просмотра
MIN_USER_INTERACTIONS = 10    # минимум интеракций на пользователя
MIN_ITEM_INTERACTIONS = 10    # минимум интеракций на айтем

df = interactions.copy()

# Оставляем только достаточно длинные просмотры
df = df[df["total_dur"] >= MIN_DUR].copy()

# Считаем, сколько интеракций у каждого пользователя
user_cnt = df.groupby("user_id")["item_id"].count()

# Оставляем только активных пользователей
good_users = user_cnt[user_cnt >= MIN_USER_INTERACTIONS].index
df = df[df["user_id"].isin(good_users)].copy()

# Считаем, сколько пользователей взаимодействовало с каждым айтемом
item_cnt = df.groupby("item_id")["user_id"].count()

# Оставляем только достаточно популярные айтемы
good_items = item_cnt[item_cnt >= MIN_ITEM_INTERACTIONS].index
df = df[df["item_id"].isin(good_items)].copy()

print("shape after filtering:", df.shape)
print("n_users after filtering:", df["user_id"].nunique())
print("n_items after filtering:", df["item_id"].nunique())

shape after filtering: (2300516, 5)
n_users after filtering: 109087
n_items after filtering: 6534


In [34]:
max_date = df["last_watch_dt"].max()

test_threshold = max_date - pd.Timedelta(days=7)
val_threshold = test_threshold - pd.Timedelta(days=60)

# test — самые свежие взаимодействия
test = df[df["last_watch_dt"] >= test_threshold].copy()

# всё, что было раньше test
train_val = df[df["last_watch_dt"] < test_threshold].copy()

# val — последние 60 дней внутри train_val
val = train_val[train_val["last_watch_dt"] >= val_threshold].copy()

# train — всё, что ещё раньше
train = train_val[train_val["last_watch_dt"] < val_threshold].copy()

# для валидации оставляем только тех пользователей, которые уже встречались в train.
# Иначе модель просто не сможет по ним рекомендовать.
val = val[val["user_id"].isin(train["user_id"].unique())].copy()

print("train shape:", train.shape)
print("val shape:", val.shape)
print("test shape:", test.shape)

train shape: (881660, 5)
val shape: (694032, 5)
test shape: (172593, 5)


In [35]:
# implicit ожидает матрицу:
#   строки    -> user_id
#   столбцы   -> item_id
#   значения  -> strength взаимодействия
#

train_sparse = sp.csr_matrix(
    (
        train["watched_pct"].astype(float).values,   # значения в матрице
        (
            train["user_id"].astype(int).values,     # индексы строк
            train["item_id"].astype(int).values      # индексы столбцов
        )
    ),
    shape=(
        df["user_id"].max() + 1,
        df["item_id"].max() + 1
    )
)

print("train_sparse shape:", train_sparse.shape)
print("train_sparse nnz:", train_sparse.nnz)  # число ненулевых элементов

train_sparse shape: (1097522, 16517)
train_sparse nnz: 881660


## BPR

In [36]:
bpr = implicit.bpr.BayesianPersonalizedRanking(
    factors=50,
    regularization=0.01,
    iterations=50,
    use_gpu=False
)

bpr.fit(train_sparse.astype("double"))

100%|██████████| 50/50 [00:04<00:00, 11.46it/s, train_auc=85.64%, skipped=3.57%]


In [37]:
# 6. Функции метрик

def precision_at_k(df_pred, pred_col="preds", true_col="true_items", k=20):
    scores = []

    for _, row in df_pred.iterrows():
        pred = row[pred_col][:k] if isinstance(row[pred_col], list) else []
        true = row[true_col] if isinstance(row[true_col], list) else []

        if len(pred) == 0:
            continue

        # сколько релевантных попало в top-k
        hits = len(set(pred) & set(true))

        # precision = hits / число предсказаний
        scores.append(hits / min(k, len(pred)))

    return float(np.mean(scores)) if scores else 0.0


def recall_at_k(df_pred, pred_col="preds", true_col="true_items", k=20):
    scores = []

    for _, row in df_pred.iterrows():
        pred = row[pred_col][:k] if isinstance(row[pred_col], list) else []
        true = row[true_col] if isinstance(row[true_col], list) else []

        if len(true) == 0:
            continue

        # сколько релевантных попало в top-k
        hits = len(set(pred) & set(true))

        # recall = hits / число истинно релевантных
        scores.append(hits / len(true))

    return float(np.mean(scores)) if scores else 0.0


def mrr_at_k(df_pred, pred_col="preds", true_col="true_items", k=20):
    scores = []

    for _, row in df_pred.iterrows():
        pred = row[pred_col][:k] if isinstance(row[pred_col], list) else []
        true = set(row[true_col]) if isinstance(row[true_col], list) else set()

        # reciprocal rank первого релевантного айтема
        rr = 0.0
        for rank, item in enumerate(pred, start=1):
            if item in true:
                rr = 1.0 / rank
                break

        scores.append(rr)

    return float(np.mean(scores)) if scores else 0.0

In [38]:
# Получение рекомендаций на validation
# Берём пользователей из val и для каждого строим top-N рекомендаций.
#
# В implicit recommend работает так:
# - userid = массив пользователей
# - user_items = матрица взаимодействий этих пользователей
# - N = сколько рекомендаций вернуть
# - filter_already_liked_items=True
#   чтобы не рекомендовать то, что уже было в train.

K = 100
val_users = val["user_id"].unique()

val_item_ids, val_scores = bpr.recommend(
    userid=val_users,
    user_items=train_sparse[val_users],
    N=K,
    filter_already_liked_items=True
)

In [39]:
# Подготовка ground truth и предсказаний
# Для каждого пользователя из validation:
# - true_items = список айтемов, которые реально были в val
# - preds = список рекомендаций от BPR

val_true = (
    val.groupby("user_id")["item_id"]
       .agg(list)
       .reset_index()
       .rename(columns={"item_id": "true_items"})
)

val_pred = pd.DataFrame({
    "user_id": val_users,
    "preds": val_item_ids.tolist()
})

# Соединяем truth и predictions в одну таблицу
val_eval = val_true.merge(val_pred, on="user_id", how="inner")

val_eval.head()

,user_id,true_items,preds
0,2,"[13867, 10770, 7106, 16029, 8482, 5819, 12449,...","[3166, 6738, 4072, 12299, 6577, 9164, 7745, 16..."
1,21,"[10878, 8251, 7571, 12995, 12184, 8821, 6384, ...","[24, 826, 6939, 14025, 12659, 147, 7557, 9817,..."
2,30,"[2743, 3031, 12418, 15363, 15845, 9842, 16484,...","[9728, 13865, 142, 6809, 12202, 16447, 10440, ..."
3,46,[10440],"[6809, 16509, 16223, 11640, 9996, 4151, 10440,..."
4,60,"[9728, 10811, 6044, 8612, 1343, 6606, 9950, 14...","[4151, 2657, 7571, 6050, 2910, 7807, 9982, 108..."


In [ ]:
# Подсчёт baseline-метрик на validation

p20 = precision_at_k(val_eval, k=20)
r20 = recall_at_k(val_eval, k=20)
mrr20 = mrr_at_k(val_eval, k=20)

print(f"precision@20 = {p20:.4f}")
print(f"recall@20    = {r20:.4f}")
print(f"mrr@20       = {mrr20:.4f}")

precision@20 = 0.0479
recall@20    = 0.1096
mrr@20       = 0.1813


## Второй уровень

In [41]:
# Генерация кандидатов от BPR для validation
# Здесь:
# - val_users = пользователи из validation
# - K_CANDIDATES = сколько кандидатов берём на второго уровня модель

K_CANDIDATES = 100
val_users = val["user_id"].unique()

# recommend возвращает:
# - val_item_ids: массив рекомендованных item_id
# - val_scores: соответствующие скоринги BPR
#
# filter_already_liked_items=True нужен, чтобы не рекомендовать
# то, что пользователь уже видел в train.[web:78]
val_item_ids, val_scores = bpr.recommend(
    userid=val_users,
    user_items=train_sparse[val_users],
    N=K_CANDIDATES,
    filter_already_liked_items=True
)

In [42]:
# Превращаем выдачу BPR в "длинную" таблицу кандидатов

candidates = pd.DataFrame({
    "user_id": val_users,
    "item_id": val_item_ids.tolist(),
    "score_bpr": val_scores.tolist()
})

# explode превращает список кандидатов в отдельные строки
candidates = candidates.explode(["item_id", "score_bpr"]).reset_index(drop=True)

# Явно приводим типы
candidates["user_id"] = candidates["user_id"].astype(int)
candidates["item_id"] = candidates["item_id"].astype(int)
candidates["score_bpr"] = candidates["score_bpr"].astype(float)

# rank_bpr = место айтема внутри списка рекомендаций пользователя
# cumcount() считает 0,1,2..., поэтому +1
candidates["rank_bpr"] = candidates.groupby("user_id").cumcount() + 1

candidates.head()

,user_id,item_id,score_bpr,rank_bpr
0,927973,4880,0.988075,1
1,927973,4151,0.964167,2
2,927973,142,0.840022,3
3,927973,2028,0.830607,4
4,927973,7417,0.722170,5


In [43]:
# Готовим "истину" для validation

val_true = (
    val.groupby("user_id")["item_id"]
       .agg(list)
       .reset_index()
       .rename(columns={"item_id": "true_items"})
)

val_true.head()

,user_id,true_items
0,2,"[13867, 10770, 7106, 16029, 8482, 5819, 12449,..."
1,21,"[10878, 8251, 7571, 12995, 12184, 8821, 6384, ..."
2,30,"[2743, 3031, 12418, 15363, 15845, 9842, 16484,..."
3,46,[10440]
4,60,"[9728, 10811, 6044, 8612, 1343, 6606, 9950, 14..."


In [44]:
# Размечаем target для кандидатов
# target = 1, если candidate item действительно был у пользователя в val
# target = 0, если это просто кандидат, но в val он не встретился

val_true_dict = {
    row["user_id"]: set(row["true_items"])
    for _, row in val_true.iterrows()
}

# Проверяем каждый кандидат:
# попал ли он в реальные item'ы пользователя на validation
candidates["target"] = candidates.apply(
    lambda x: int(x["item_id"] in val_true_dict.get(x["user_id"], set())),
    axis=1
)

candidates.head(10)

,user_id,item_id,score_bpr,rank_bpr,target
0,927973,4880,0.988075,1,1
1,927973,4151,0.964167,2,1
2,927973,142,0.840022,3,0
3,927973,2028,0.830607,4,0
4,927973,7417,0.722170,5,0
5,927973,7807,0.717595,6,0
6,927973,10440,0.703339,7,0
7,927973,6809,0.701622,8,0
8,927973,1261,0.701191,9,0
9,927973,15297,0.701079,10,1


In [45]:
# Смотрим, сколько получилось позитивов и негативов

print(candidates["target"].value_counts(dropna=False))
print("positive rate:", candidates["target"].mean())

target
0    6134909
1     127091
Name: count, dtype: int64
positive rate: 0.020295592462472055


In [47]:
# Добавляем простые фичи на основе BPR-выдачи

candidates["rank_inv"] = 1.0 / candidates["rank_bpr"]

candidates.head()

,user_id,item_id,score_bpr,rank_bpr,target,rank_inv
0,927973,4880,0.988075,1,1,1.000000
1,927973,4151,0.964167,2,1,0.500000
2,927973,142,0.840022,3,0,0.333333
3,927973,2028,0.830607,4,0,0.250000
4,927973,7417,0.722170,5,0,0.200000


In [48]:
# Добавляем признаки пользователей

user_features = users[[
    "user_id",
    "age",
    "income",
    "sex",
    "kids_flg"
]].copy()

candidates = candidates.merge(user_features, on="user_id", how="left")

candidates.head()

,user_id,item_id,score_bpr,rank_bpr,target,rank_inv,age,income,sex,kids_flg
0,927973,4880,0.988075,1,1,1.000000,age_25_34,income_20_40,М,0.0
1,927973,4151,0.964167,2,1,0.500000,age_25_34,income_20_40,М,0.0
2,927973,142,0.840022,3,0,0.333333,age_25_34,income_20_40,М,0.0
3,927973,2028,0.830607,4,0,0.250000,age_25_34,income_20_40,М,0.0
4,927973,7417,0.722170,5,0,0.200000,age_25_34,income_20_40,М,0.0


In [49]:
# Добавляем признаки айтемов

item_features = items[[
    "item_id",
    "content_type",
    "countries",
    "for_kids",
    "age_rating",
    "studios"
]].copy()

candidates = candidates.merge(item_features, on="item_id", how="left")

candidates.head()

,user_id,item_id,score_bpr,rank_bpr,target,rank_inv,age,income,sex,kids_flg,content_type,countries,for_kids,age_rating,studios
0,927973,4880,0.988075,1,1,1.000000,age_25_34,income_20_40,М,0.0,series,Россия,NaN,18.0,NaN
1,927973,4151,0.964167,2,1,0.500000,age_25_34,income_20_40,М,0.0,series,Россия,NaN,18.0,NaN
2,927973,142,0.840022,3,0,0.333333,age_25_34,income_20_40,М,0.0,film,Россия,NaN,16.0,NaN
3,927973,2028,0.830607,4,0,0.250000,age_25_34,income_20_40,М,0.0,film,Россия,NaN,12.0,NaN
4,927973,7417,0.722170,5,0,0.200000,age_25_34,income_20_40,М,0.0,film,Россия,NaN,16.0,NaN


In [50]:
# Дополнительные агрегатные фичи из train
# user_watch_cnt      — сколько взаимодействий было у пользователя в train
# item_popularity     — сколько пользователей взаимодействовало с item в train
# item_avg_watched_pct — средний watched_pct по item в train


user_watch_cnt = (
    train.groupby("user_id")["item_id"]
         .count()
         .reset_index()
         .rename(columns={"item_id": "user_watch_cnt"})
)

item_popularity = (
    train.groupby("item_id")["user_id"]
         .count()
         .reset_index()
         .rename(columns={"user_id": "item_popularity"})
)

item_avg_watched_pct = (
    train.groupby("item_id")["watched_pct"]
         .mean()
         .reset_index()
         .rename(columns={"watched_pct": "item_avg_watched_pct"})
)

candidates = (
    candidates
    .merge(user_watch_cnt, on="user_id", how="left")
    .merge(item_popularity, on="item_id", how="left")
    .merge(item_avg_watched_pct, on="item_id", how="left")
)

candidates.head()

,user_id,item_id,score_bpr,rank_bpr,target,rank_inv,age,income,sex,kids_flg,content_type,countries,for_kids,age_rating,studios,user_watch_cnt,item_popularity,item_avg_watched_pct
0,927973,4880,0.988075,1,1,1.000000,age_25_34,income_20_40,М,0.0,series,Россия,NaN,18.0,NaN,3,10414,12.831957
1,927973,4151,0.964167,2,1,0.500000,age_25_34,income_20_40,М,0.0,series,Россия,NaN,18.0,NaN,3,12304,59.150927
2,927973,142,0.840022,3,0,0.333333,age_25_34,income_20_40,М,0.0,film,Россия,NaN,16.0,NaN,3,9452,81.224185
3,927973,2028,0.830607,4,0,0.250000,age_25_34,income_20_40,М,0.0,film,Россия,NaN,12.0,NaN,3,247,77.744939
4,927973,7417,0.722170,5,0,0.200000,age_25_34,income_20_40,М,0.0,film,Россия,NaN,16.0,NaN,3,4059,80.078837


In [52]:
feature_cols = [
    # сигналы от первого уровня
    "rank_bpr",
    "score_bpr",
    "rank_inv",

    # user features
    "age",
    "income",
    "sex",
    "kids_flg",

    # item features
    "content_type",
    "countries",
    "for_kids",
    "age_rating",
    "studios",

    # агрегаты из train
    "user_watch_cnt",
    "item_popularity",
    "item_avg_watched_pct"
]

cat_cols = [
    "age",
    "income",
    "sex",
    "content_type",
    "countries",
    "studios"
]

target_col = "target"

# Оставляем только нужные колонки
rank_df = candidates[["user_id", "item_id", target_col] + feature_cols].copy()

rank_df.head()

,user_id,item_id,target,rank_bpr,score_bpr,rank_inv,age,income,sex,kids_flg,content_type,countries,for_kids,age_rating,studios,user_watch_cnt,item_popularity,item_avg_watched_pct
0,927973,4880,1,1,0.988075,1.000000,age_25_34,income_20_40,М,0.0,series,Россия,NaN,18.0,NaN,3,10414,12.831957
1,927973,4151,1,2,0.964167,0.500000,age_25_34,income_20_40,М,0.0,series,Россия,NaN,18.0,NaN,3,12304,59.150927
2,927973,142,0,3,0.840022,0.333333,age_25_34,income_20_40,М,0.0,film,Россия,NaN,16.0,NaN,3,9452,81.224185
3,927973,2028,0,4,0.830607,0.250000,age_25_34,income_20_40,М,0.0,film,Россия,NaN,12.0,NaN,3,247,77.744939
4,927973,7417,0,5,0.722170,0.200000,age_25_34,income_20_40,М,0.0,film,Россия,NaN,16.0,NaN,3,4059,80.078837


In [53]:
# Заполняем пропуски

for col in feature_cols:
    if col in cat_cols:
        rank_df[col] = rank_df[col].fillna("unknown").astype(str)
    else:
        rank_df[col] = rank_df[col].fillna(rank_df[col].median())

rank_df.head()

,user_id,item_id,target,rank_bpr,score_bpr,rank_inv,age,income,sex,kids_flg,content_type,countries,for_kids,age_rating,studios,user_watch_cnt,item_popularity,item_avg_watched_pct
0,927973,4880,1,1,0.988075,1.000000,age_25_34,income_20_40,М,0.0,series,Россия,0.0,18.0,unknown,3,10414,12.831957
1,927973,4151,1,2,0.964167,0.500000,age_25_34,income_20_40,М,0.0,series,Россия,0.0,18.0,unknown,3,12304,59.150927
2,927973,142,0,3,0.840022,0.333333,age_25_34,income_20_40,М,0.0,film,Россия,0.0,16.0,unknown,3,9452,81.224185
3,927973,2028,0,4,0.830607,0.250000,age_25_34,income_20_40,М,0.0,film,Россия,0.0,12.0,unknown,3,247,77.744939
4,927973,7417,0,5,0.722170,0.200000,age_25_34,income_20_40,М,0.0,film,Россия,0.0,16.0,unknown,3,4059,80.078837


In [54]:
# Делим пользователей на train/eval для reranker
# делить нужно по user_id, а не по строкам.
# Иначе один и тот же пользователь попадёт и в train, и в eval,
# и мы получим утечку.
#
# Берём, например, 80% пользователей на train
# и 20% пользователей на eval.

RANDOM_STATE = 42
EVAL_FRAC = 0.2

all_rank_users = rank_df["user_id"].drop_duplicates().values
rng = np.random.RandomState(RANDOM_STATE)

eval_users = rng.choice(
    all_rank_users,
    size=int(len(all_rank_users) * EVAL_FRAC),
    replace=False
)

eval_users = set(eval_users)
train_users = set(all_rank_users) - eval_users

rank_train = rank_df[rank_df["user_id"].isin(train_users)].copy()
rank_eval = rank_df[rank_df["user_id"].isin(eval_users)].copy()

print("rank_train shape:", rank_train.shape)
print("rank_eval shape:", rank_eval.shape)

rank_train shape: (5009600, 18)
rank_eval shape: (1252400, 18)


In [ ]:
# Для ranking-задач CatBoost требует, чтобы объекты одной группы
# шли подряд в датасете.


rank_train = rank_train.sort_values(["user_id", "rank_bpr"]).reset_index(drop=True)
rank_eval = rank_eval.sort_values(["user_id", "rank_bpr"]).reset_index(drop=True)

In [56]:
# Готовим X / y / group_id для CatBoostRanker

X_train = rank_train[feature_cols].copy()
y_train = rank_train[target_col].copy()
group_id_train = rank_train["user_id"].copy()

X_eval = rank_eval[feature_cols].copy()
y_eval = rank_eval[target_col].copy()
group_id_eval = rank_eval["user_id"].copy()

print(X_train.shape, X_eval.shape)

(5009600, 15) (1252400, 15)


In [57]:
from catboost import Pool, CatBoostRanker


In [58]:
# Подготовка Pool для CatBoostRanker

train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=group_id_train,
    cat_features=cat_cols
)

eval_pool = Pool(
    data=X_eval,
    label=y_eval,
    group_id=group_id_eval,
    cat_features=cat_cols
)

In [59]:
ranker = CatBoostRanker(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="YetiRankPairwise",
    eval_metric="NDCG:top=20",
    random_seed=42,
    verbose=50
)

ranker.fit(
    train_pool,
    eval_set=eval_pool,
    use_best_model=True
)

Pairwise scoring loss functions on CPU do not support one hot features. OneHotMaxSize set to 1
0:	test: 0.5729652	best: 0.5729652 (0)	total: 1.55s	remaining: 12m 51s
50:	test: 0.6423412	best: 0.6424410 (49)	total: 59s	remaining: 8m 39s
100:	test: 0.6606122	best: 0.6606122 (100)	total: 2m	remaining: 7m 55s
150:	test: 0.6654406	best: 0.6654950 (149)	total: 3m 4s	remaining: 7m 5s
200:	test: 0.6668345	best: 0.6669061 (195)	total: 4m 8s	remaining: 6m 9s
250:	test: 0.6681723	best: 0.6681752 (249)	total: 5m 14s	remaining: 5m 12s
300:	test: 0.6691319	best: 0.6691319 (300)	total: 6m 21s	remaining: 4m 11s
350:	test: 0.6695781	best: 0.6696993 (335)	total: 7m 25s	remaining: 3m 9s
400:	test: 0.6703406	best: 0.6703406 (400)	total: 8m 30s	remaining: 2m 6s
450:	test: 0.6716322	best: 0.6716624 (448)	total: 9m 34s	remaining: 1m 2s
499:	test: 0.6726264	best: 0.6726264 (499)	total: 10m 36s	remaining: 0us

bestTest = 0.6726264324
bestIteration = 499



CatBoostRanker(depth=6, eval_metric='NDCG:top=20', iterations=500, learning_rate=0.05, loss_function='YetiRankPairwise', random_seed=42, verbose=50)

In [60]:
rank_eval = rank_eval.copy()
rank_eval["ranker_score"] = ranker.predict(X_eval)

rank_eval.head()

,user_id,item_id,target,rank_bpr,score_bpr,rank_inv,age,income,sex,kids_flg,content_type,countries,for_kids,age_rating,studios,user_watch_cnt,item_popularity,item_avg_watched_pct,ranker_score
0,30,9728,1,1,0.753296,1.000000,unknown,unknown,unknown,0.0,film,"Великобритания, США",0.0,18.0,unknown,2,295,96.610169,3.011390
1,30,13865,0,2,0.750278,0.500000,unknown,unknown,unknown,0.0,film,Россия,0.0,12.0,unknown,2,11111,82.251553,2.813726
2,30,142,0,3,0.726561,0.333333,unknown,unknown,unknown,0.0,film,Россия,0.0,16.0,unknown,2,9452,81.224185,2.183506
3,30,6809,0,4,0.719728,0.250000,unknown,unknown,unknown,0.0,film,Россия,0.0,16.0,unknown,2,4956,61.914044,1.343648
4,30,12202,0,5,0.704795,0.200000,unknown,unknown,unknown,0.0,film,Россия,0.0,6.0,unknown,2,134,59.947761,-0.304733


In [61]:

rank_eval_sorted = (
    rank_eval
    .sort_values(["user_id", "ranker_score"], ascending=[True, False])
    .reset_index(drop=True)
)

# Собираем финальный список айтемов на пользователя
eval_pred_ranker = (
    rank_eval_sorted
    .groupby("user_id")["item_id"]
    .agg(list)
    .reset_index()
    .rename(columns={"item_id": "preds"})
)

eval_pred_ranker.head()

,user_id,preds
0,30,"[15297, 10440, 9728, 13865, 2657, 4151, 142, 4..."
1,322,"[12995, 4457, 1287, 16361, 15915, 416, 4702, 6..."
2,340,"[15297, 10440, 2657, 4880, 9996, 8636, 12995, ..."
3,467,"[7571, 13915, 13243, 5051, 13545, 4718, 9081, ..."
4,473,"[7571, 11310, 13018, 13915, 9811, 14414, 9709,..."


In [62]:
eval_true = (
    val[val["user_id"].isin(eval_users)]
    .groupby("user_id")["item_id"]
    .agg(list)
    .reset_index()
    .rename(columns={"item_id": "true_items"})
)

eval_true.head()

,user_id,true_items
0,30,"[2743, 3031, 12418, 15363, 15845, 9842, 16484,..."
1,322,"[15163, 142, 10772, 3006, 15297, 15603, 10440,..."
2,340,"[10440, 15297, 8618, 6443, 5311, 2657]"
3,467,[9728]
4,473,"[9728, 7582, 8636, 9937, 11699, 8026, 4976, 44..."


In [63]:
eval_result_ranker = eval_true.merge(
    eval_pred_ranker,
    on="user_id",
    how="inner"
)

eval_result_ranker.head()

,user_id,true_items,preds
0,30,"[2743, 3031, 12418, 15363, 15845, 9842, 16484,...","[15297, 10440, 9728, 13865, 2657, 4151, 142, 4..."
1,322,"[15163, 142, 10772, 3006, 15297, 15603, 10440,...","[12995, 4457, 1287, 16361, 15915, 416, 4702, 6..."
2,340,"[10440, 15297, 8618, 6443, 5311, 2657]","[15297, 10440, 2657, 4880, 9996, 8636, 12995, ..."
3,467,[9728],"[7571, 13915, 13243, 5051, 13545, 4718, 9081, ..."
4,473,"[9728, 7582, 8636, 9937, 11699, 8026, 4976, 44...","[7571, 11310, 13018, 13915, 9811, 14414, 9709,..."


In [64]:
p20_ranker = precision_at_k(eval_result_ranker, k=20)
r20_ranker = recall_at_k(eval_result_ranker, k=20)
mrr20_ranker = mrr_at_k(eval_result_ranker, k=20)

print(f"Ranker precision@20 = {p20_ranker:.4f}")
print(f"Ranker recall@20    = {r20_ranker:.4f}")
print(f"Ranker mrr@20       = {mrr20_ranker:.4f}")

Ranker precision@20 = 0.0692
Ranker recall@20    = 0.1579
Ranker mrr@20       = 0.3979


In [65]:
test = test[test["user_id"].isin(train["user_id"].unique())].copy()

test_users = test["user_id"].unique()
print("test users:", len(test_users))

test users: 23871


In [66]:
K_CANDIDATES_TEST = 100

test_item_ids_bpr, test_scores_bpr = bpr.recommend(
    userid=test_users,
    user_items=train_sparse[test_users],
    N=K_CANDIDATES_TEST,
    filter_already_liked_items=True
)

test_candidates = pd.DataFrame({
    "user_id": test_users,
    "item_id": test_item_ids_bpr.tolist(),
    "score_bpr": test_scores_bpr.tolist()
})

test_candidates = test_candidates.explode(["item_id", "score_bpr"]).reset_index(drop=True)
test_candidates["user_id"] = test_candidates["user_id"].astype(int)
test_candidates["item_id"] = test_candidates["item_id"].astype(int)
test_candidates["score_bpr"] = test_candidates["score_bpr"].astype(float)
test_candidates["rank_bpr"] = test_candidates.groupby("user_id").cumcount() + 1
test_candidates["rank_inv"] = 1.0 / test_candidates["rank_bpr"]

In [67]:
test_candidates = (
    test_candidates
    .merge(user_features, on="user_id", how="left")
    .merge(item_features, on="item_id", how="left")
    .merge(user_watch_cnt, on="user_id", how="left")
    .merge(item_popularity, on="item_id", how="left")
    .merge(item_avg_watched_pct, on="item_id", how="left")
)

# заполнение пропусков по тем же правилам
for col in feature_cols:
    if col in cat_cols:
        test_candidates[col] = test_candidates[col].fillna("unknown").astype(str)
    else:
        test_candidates[col] = test_candidates[col].fillna(rank_df[col].median())

In [69]:
X_test_rank = test_candidates[feature_cols].copy()
test_candidates["ranker_score"] = ranker.predict(X_test_rank)

# сортируем кандидатов по score внутри каждого user_id
test_rank_sorted = (
    test_candidates
    .sort_values(["user_id", "ranker_score"], ascending=[True, False])
    .reset_index(drop=True)
)

# собираем итоговые списки рекомендаций
test_pred_ranker = (
    test_rank_sorted
    .groupby("user_id")["item_id"]
    .agg(list)
    .reset_index()
    .rename(columns={"item_id": "preds"}))

In [70]:
test_true = (
    test.groupby("user_id")["item_id"]
        .agg(list)
        .reset_index()
        .rename(columns={"item_id": "true_items"})
)

test_result_ranker = test_true.merge(test_pred_ranker, on="user_id", how="inner")

print("Test ranker precision@20:", precision_at_k(test_result_ranker, k=20))
print("Test ranker recall@20:", recall_at_k(test_result_ranker, k=20))
print("Test ranker mrr@20:", mrr_at_k(test_result_ranker, k=20))

Test ranker precision@20: 0.009643500481756107
Test ranker recall@20: 0.07688845003001549
Test ranker mrr@20: 0.05778283111027203
